# 03 — Procesamiento Distribuido con Modin (Dask Distributed — Dataproc)

Versión distribuida real. Modin usa el **Dask Scheduler** como backend — cada worker procesa un año completo desde GCS y devuelve solo los agregados al master.

**Flujo:** GCS raw/ (2017–2025) → `client.map` (workers Modin/Dask) → agregados → master combina → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Tendencia mensual de crímenes domésticos por era | `domestic_trend` |
| 2 | **Read**   | Top 10 bloques con más crímenes por era | `top_blocks` |
| 3 | **Read**   | Distribución de crímenes por día de la semana | — (exploración) |
| 4 | **Update** | Añadir `location_category` a registros | `location_categories` |
| 5 | **Delete** | Eliminar registros con `primary_type` nulo | — (limpieza) |

In [1]:
from dask.distributed import Client
import pandas as pd
import time
import os

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}

def to_bigquery(df, table_name):
    df.to_gbq(destination_table=f'{DATASET_ID}.{table_name}',
              project_id=PROJECT_ID, if_exists='replace', progress_bar=False)
    print(f'  → BigQuery: {DATASET_ID}.{table_name}  ({len(df):,} filas)')

client = Client('localhost:8786')
print(client)
print(f'Workers activos: {len(client.scheduler_info()["workers"])}')
for wid, w in client.scheduler_info()['workers'].items():
    print(f'  {wid}  ncores={w["nthreads"]}')

<Client: 'tcp://10.128.0.56:8786' processes=2 threads=8, memory=22.35 GiB>
Workers activos: 2
  tcp://10.128.0.55:42759  ncores=4
  tcp://10.128.0.57:43975  ncores=4


In [2]:
# ── Función que corre completamente en los workers ────────────────────────────
# Patrón Modin[Dask]: cada worker aplica operaciones pandas sobre su partición (1 año)
# Solo los resultados agregados (KB) viajan de vuelta al master
def process_year_modin(year):
    import pandas as pd
    import gcsfs

    ERA = {2017:'PRE', 2018:'PRE', 2019:'PRE',
           2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
           2023:'POST', 2024:'POST', 2025:'POST'}
    OUTDOOR    = {'STREET','SIDEWALK','ALLEY','PARK PROPERTY',
                  'PARKING LOT / GARAGE (NON RESIDENTIAL)','SCHOOL, PUBLIC, GROUNDS'}
    INDOOR     = {'RESIDENCE','APARTMENT','RESIDENCE - GARAGE',
                  'RESIDENCE PORCH/HALLWAY','SCHOOL, PUBLIC, BUILDING'}
    COMMERCIAL = {'RESTAURANT','SMALL RETAIL STORE','GAS STATION',
                  'GROCERY FOOD STORE','CONVENIENCE STORE','CURRENCY EXCHANGE'}

    # Leer desde GCS en el worker
    fs = gcsfs.GCSFileSystem(project='my-first-project-492901', token='google_default')
    with fs.open(f'big-data-proyecto-parcial/raw/Chicago_Crimes_{year}.csv') as f:
        df = pd.read_csv(f, dtype={
            'unique_key':'Int64','beat':'Int64','district':'Int64',
            'ward':'Int64','community_area':'Int64',
            'x_coordinate':'Int64','y_coordinate':'Int64','year':'Int64',
            'latitude':'float64','longitude':'float64',
        }, parse_dates=['date'], on_bad_lines='skip')

    df['covid_era']   = df['year'].map(ERA)
    df['month']       = df['date'].dt.month
    df['year_num']    = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.dayofweek

    def classify(loc):
        if pd.isna(loc): return 'UNKNOWN'
        u = str(loc).upper()
        if u in OUTDOOR:    return 'OUTDOOR'
        if u in INDOOR:     return 'INDOOR'
        if u in COMMERCIAL: return 'COMMERCIAL'
        return 'OTHER'

    df['location_category'] = df['location_description'].apply(classify)

    # CRUD 1 — domestic trend
    domestic = df[df['domestic'] == True]
    dom_trend = domestic.groupby(['year_num','month','covid_era']).agg(
        domestic_crimes=('unique_key','count'), arrests=('arrest','sum')).reset_index()

    # CRUD 2 — top blocks
    top_blk = df.groupby(['block','covid_era']).agg(
        total_crimes=('unique_key','count'), arrests=('arrest','sum'),
        distinct_types=('primary_type','nunique')).reset_index()

    # CRUD 3 — weekly
    weekly = df.groupby(['day_of_week','covid_era']).agg(
        total_crimes=('unique_key','count'), arrests=('arrest','sum')).reset_index()

    # CRUD 4 — location category
    loc_dist = df.groupby(['covid_era','location_category']).size().reset_index(name='count')

    # CRUD 5 — missing primary_type
    n_total   = len(df)
    n_missing = int(df['primary_type'].isna().sum())
    n_valid   = int(df['primary_type'].notna().sum())

    return {
        'year': year, 'n_total': n_total, 'n_missing_type': n_missing, 'n_valid': n_valid,
        'dom_trend': dom_trend, 'top_blk': top_blk, 'weekly': weekly, 'loc_dist': loc_dist,
    }

t0 = time.time()
print('Distribuyendo procesamiento Modin a workers...')
futures = client.map(process_year_modin, YEARS_ANALYSIS)
results = client.gather(futures)   # solo trae agregados pequeños
elapsed = time.time() - t0

total = sum(r['n_total'] for r in results)
print(f'Procesamiento distribuido completado en {elapsed:.1f}s')
print(f'Registros procesados: {total:,}')
print(f'Workers activos:      {len(client.scheduler_info()["workers"])}')
for wid, w in client.scheduler_info()['workers'].items():
    print(f'  {wid}  ncores={w["nthreads"]}')

Distribuyendo procesamiento Modin a workers...
Procesamiento distribuido completado en 15.1s
Registros procesados: 2,072,943
Workers activos:      2
  tcp://10.128.0.55:42759  ncores=4
  tcp://10.128.0.57:43975  ncores=4


---
## CRUD 1 — CREATE: Tendencia mensual de crímenes domésticos

In [3]:
dom_trend = (
    pd.concat([r['dom_trend'] for r in results], ignore_index=True)
    .groupby(['year_num','month','covid_era'])
    .agg(domestic_crimes=('domestic_crimes','sum'), arrests=('arrests','sum'))
    .reset_index()
    .sort_values(['year_num','month'])
)
dom_trend['arrest_rate_pct'] = (dom_trend['arrests'] / dom_trend['domestic_crimes'] * 100).round(2)

print('Crímenes domésticos por era COVID:')
era_summary = dom_trend.groupby('covid_era')['domestic_crimes'].sum()
for era in ['PRE','DURANTE','POST']:
    print(f'  {era:<8}: {era_summary.get(era, 0):,}')

print(f'Workers activos: {len(client.scheduler_info()["workers"])}')
to_bigquery(dom_trend[['year_num','month','covid_era','domestic_crimes','arrests','arrest_rate_pct']], 'domestic_trend')

Crímenes domésticos por era COVID:
  PRE     : 153,534
  DURANTE : 139,095
  POST    : 112,471
Workers activos: 2
  → BigQuery: chicago_crimes_results.domestic_trend  (102 filas)


---
## CRUD 2 — READ: Top 10 bloques con más crímenes

In [4]:
top_blocks = (
    pd.concat([r['top_blk'] for r in results], ignore_index=True)
    .groupby(['block','covid_era'])
    .agg(total_crimes=('total_crimes','sum'), arrests=('arrests','sum'),
         distinct_types=('distinct_types','max'))
    .reset_index()
)
top10_era = (
    top_blocks.groupby('covid_era', group_keys=False)
    .apply(lambda x: x.nlargest(10,'total_crimes'))
    .reset_index(drop=True)
)

print('Top 3 bloques por era COVID:')
for era in ['PRE','DURANTE','POST']:
    sub = top10_era[top10_era['covid_era'] == era].head(3)
    for _, row in sub.iterrows():
        print(f'  {era}: {row["block"][:45]}  → {row["total_crimes"]:,}')

to_bigquery(top10_era, 'top_blocks')

Top 3 bloques por era COVID:
  PRE: 001XX N STATE ST  → 3,015
  PRE: 008XX N MICHIGAN AVE  → 1,509
  PRE: 0000X W TERMINAL ST  → 1,293
  DURANTE: 001XX N STATE ST  → 1,452
  DURANTE: 0000X W TERMINAL ST  → 1,282
  DURANTE: 033XX W FILLMORE ST  → 907
  POST: 001XX N STATE ST  → 1,790
  POST: 0000X W TERMINAL ST  → 1,197
  POST: 0000X N STATE ST  → 1,129
  → BigQuery: chicago_crimes_results.top_blocks  (30 filas)


---
## CRUD 3 — READ: Distribución por día de la semana

In [5]:
DAY_NAMES = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',
             4:'Friday',5:'Saturday',6:'Sunday'}

weekly = (
    pd.concat([r['weekly'] for r in results], ignore_index=True)
    .groupby(['day_of_week','covid_era'])
    .agg(total_crimes=('total_crimes','sum'), arrests=('arrests','sum'))
    .reset_index()
)
weekly['day_name'] = weekly['day_of_week'].map(DAY_NAMES)
weekly['arrest_rate_pct'] = (weekly['arrests'] / weekly['total_crimes'] * 100).round(2)

print('Crímenes por día de la semana por era COVID:')
pivot_w = weekly.pivot_table(index='day_name', columns='covid_era', values='total_crimes')
print(pivot_w[['PRE','DURANTE','POST']].to_string())
print(f'\nWorkers activos: {len(client.scheduler_info()["workers"])}')

Crímenes por día de la semana por era COVID:
covid_era       PRE  DURANTE     POST
day_name                             
Friday     119463.0  97564.0  88836.0
Monday     114444.0  94453.0  87832.0
Saturday   116184.0  96234.0  88686.0
Sunday     111913.0  94404.0  88215.0
Thursday   112420.0  92396.0  85150.0
Tuesday    112917.0  92305.0  86825.0
Wednesday  112498.0  94227.0  85977.0

Workers activos: 2


---
## CRUD 4 — UPDATE: Añadir `location_category`

In [6]:
loc_dist = (
    pd.concat([r['loc_dist'] for r in results], ignore_index=True)
    .groupby(['covid_era','location_category'])['count'].sum()
    .reset_index()
)
era_totals = loc_dist.groupby('covid_era')['count'].transform('sum')
loc_dist['percentage'] = (loc_dist['count'] / era_totals * 100).round(2)

print('Distribución por categoría de ubicación y era COVID:')
pivot_loc = loc_dist.pivot_table(index='location_category', columns='covid_era', values='percentage')
print(pivot_loc[['PRE','DURANTE','POST']].to_string())

to_bigquery(loc_dist, 'location_categories')

Distribución por categoría de ubicación y era COVID:
covid_era            PRE  DURANTE   POST
location_category                       
COMMERCIAL          8.91     7.79   8.68
INDOOR             32.88    35.89  31.87
OTHER              24.93    18.33  19.30
OUTDOOR            32.80    37.39  39.73
UNKNOWN             0.47     0.60   0.42
  → BigQuery: chicago_crimes_results.location_categories  (15 filas)


---
## CRUD 5 — DELETE: Eliminar registros con `primary_type` nulo

In [7]:
print('Registros sin primary_type por año y era COVID:')
for r in results:
    era = ERA_MAP[r['year']]
    pct = r['n_missing_type'] / r['n_total'] * 100 if r['n_total'] else 0
    print(f'  {r["year"]} [{era}]: total={r["n_total"]:,}  sin_tipo={r["n_missing_type"]:,}  ({pct:.3f}%)')

n_valid = sum(r['n_valid'] for r in results)
n_total = sum(r['n_total'] for r in results)
print(f'\nRegistros originales:           {n_total:,}')
print(f'Registros válidos (post-DELETE):{n_valid:,}')
print(f'Registros eliminados:           {n_total - n_valid:,}')
print(f'Workers utilizados: {len(client.scheduler_info()["workers"])}')

client.close()
print('\nCliente Dask cerrado. Resultados guardados en BigQuery.')

Registros sin primary_type por año y era COVID:
  2017 [PRE]: total=269,214  sin_tipo=0  (0.000%)
  2018 [PRE]: total=269,070  sin_tipo=0  (0.000%)
  2019 [PRE]: total=261,555  sin_tipo=0  (0.000%)
  2020 [DURANTE]: total=212,522  sin_tipo=0  (0.000%)
  2021 [DURANTE]: total=209,406  sin_tipo=0  (0.000%)
  2022 [DURANTE]: total=239,655  sin_tipo=0  (0.000%)
  2023 [POST]: total=262,756  sin_tipo=0  (0.000%)
  2024 [POST]: total=256,305  sin_tipo=0  (0.000%)
  2025 [POST]: total=92,460  sin_tipo=0  (0.000%)

Registros originales:           2,072,943
Registros válidos (post-DELETE):2,072,943
Registros eliminados:           0
Workers utilizados: 2

Cliente Dask cerrado. Resultados guardados en BigQuery.
